# M32 · HYPOINVERSE through PocketQuake: a production locator, its files and its residuals

**Week 4 · about 7 h** · before this: M21, M30, M31

> **한국어 요약** 연구실 처리 도구 PocketQuake 안에서 HYPOINVERSE를 실행하고, 실행 기록과 결과 파일(.sum, .arc)을 직접 읽습니다. 상주 지진 6개의 진원을 기상청 목록, M31의 내 진원과 비교하고, 속도 모델을 kim1983에서 kim2011로 바꾸면 진원이 어떻게 달라지는지 봅니다. 마지막으로 관측소별 잔차의 평균으로 일관되게 이르거나 늦은 관측소를 찾습니다.

**You will**

- Run PocketQuake's location stage and check its run log
- Read HYPOINVERSE's archive file by its fixed columns
- Compare HYPOINVERSE, KMA and your own locations
- Compare two velocity models and find early and late stations

**New words:** archive file (HYPOINVERSE; 아카이브 파일), summary file (HYPOINVERSE; 요약 파일), JSON Lines (JSON 줄 형식), checksum (SHA-256 fingerprint; 체크섬), location errors (ERH and ERZ; 위치 오차), location quality (진원 결정 품질 등급), station delay (station correction; 관측소 보정값), phase misidentification (위상 오인) · [all terms](../../../docs/glossary.md)

**Optional reading:** Klein (2002), weighting and the archive format

Log any AI help in `work/ai_log/M32.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M32.yaml](../../../.sgtl/criteria/M32.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download (M03) in the data folder
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster of M21
PQ_CORES = 1                              # cores per run (at most 4 on the shared server)
PQ_MEM_GB = 8.0                           # memory cap per run in GB (the picker needs about 3.5)
MAX_DT_S = 5.0                            # one earthquake in two lists: origin times at most this far apart (s)
M31_FILE = "../M31_location/results/M31_hypoinverse_runs.csv"   # your location table of M31
M31_EVENT_ID = "2019009762"               # the earthquake of that table (KMA id)
MIN_EVENTS = 4                            # task 5: fewest readings for a station term
FIGURE_DIR = "figures"                    # your figures go here
RESULT_DIR = "results"                    # your result files go here

In [ ]:
import sgtl_tutorial as st
S = st.init("M32")

import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
HYP_DIR = RUN_DIR / "1.HypoInv"                                     # what its location stage writes
print("PocketQuake results of your cluster:", RUN_DIR)

## Part 0 · Your cluster and the six earthquakes (about 15 min)

This module locates the picks of your PocketQuake cluster of M21 (`PQ_SLUG`). From Part 4 on, it also needs your
location table of M31. The cell lists the six Sangju earthquakes of your STP download from the KMA catalog (in UTC),
and checks your cluster's picks.

In [ ]:
if not (st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py").is_file() or \
        not list((RUN_DIR / "picks").glob("*_picks.csv")):
    raise FileNotFoundError(f"PocketQuake has no picked cluster {PQ_SLUG!r} under {RUN_DIR.parent}. This module "
                            "locates the picks of your PocketQuake cluster of module M21: finish task M21-T4 of module "
                            "M21 first (or set PQ_SLUG to the cluster you used there).")
SAC_ROOT = st.data.root() / DATASET / "stp_download" / "SAC"
EVENT_FOLDERS = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir())
kma = st.data.sequence("sangju2019")
names = kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S")
six = kma[names.isin(EVENT_FOLDERS)].reset_index(drop=True)
print(f"{len(EVENT_FOLDERS)} earthquakes in your download; {len(six)} of them in the KMA catalog subset "
      f"'sangju2019' ({len(kma)} events within {kma.attrs['sequence']['radius_km']} km, 2018-2022):")
print(six[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))
print(f"\nyour cluster {PQ_SLUG!r}: {len(list((RUN_DIR / 'picks').glob('*_picks.csv')))} pick files")

KMA reports epicenters to 0.01° and depths to whole kilometers. A difference smaller than that rounding means
nothing.

## Part 1 · What HYPOINVERSE does (about 30 min)

In M31 you ran HYPOINVERSE, the U.S. Geological Survey's locator, on your own picks of one earthquake. Here
PocketQuake writes the input from its PhaseNet+ picks and runs HYPOINVERSE for every earthquake of your cluster. The
next cell prints PocketQuake's settings from your copy.

In [ ]:
PIPE = st.pq.pipeline_dir() / "pipeline"


def show_lines(path, words, last=None):
    """Print the lines of a file of your PocketQuake copy that contain any of the words (with line numbers)."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    for number, line in enumerate(lines[:last], 1):
        if any(w in line for w in words):
            print(f"{Path(path).name}:{number}: {line}")


show_lines(PIPE / "clusters" / "_base.py", ["STD_HYP =", "RMS=(4", "KIM1983 =", "KIM2011 =", "=((", "s=(("])
show_lines(PIPE / "config.py", ["(0.90, 0)", "(0.50, 2)", "LET: tuple"])
print()
show_lines(PIPE / "core" / "hypoinverse.py", ["CRH 1", "CRH 2", "DIS {DIS}", "RMS {RMS}", "ZTR {ZTR0}",
                                              "SUM '", "ARC '"])
print("\nYour cluster module (made by st.pq.stage in M21):")
print((PIPE / "clusters" / f"{PQ_SLUG}.py").read_text(encoding="utf-8").split('"""', 2)[-1].strip())

**What the printed lines mean.**

- **Weight codes** come from the PhaseNet+ probability: 0 from 0.9, 1 from 0.7, 2 from 0.5, 3 from 0.3, 4 below.
  Codes 0-3 get the weights 1, 0.75, 0.5 and 0.25; code 4 gets none.
- `KIM1983` and `KIM2011` list the speed and top depth of each layer. Stations sit on the top surface: their
  heights are not used.
- `ZTR 10 F`: each location starts at a trial depth of 10 km.

- `DIS 4 50 1 3`, distance weighting: from iteration 4, far stations lose weight, and beyond 3 × D they get none
  (D: the distance to the second-closest station, at least 50 km).
- `RMS 4 0.12 2 4`, residual weighting: from iteration 4, a reading loses weight when its residual exceeds 2 × the
  RMS. Beyond 4 × the RMS it gets none.

**Watch out:** HYPOINVERSE predicts the first arrival: the direct wave, or beyond the crossover distance a head wave
along a crustal boundary (M31 printed yours). The lab never locates with Pn or Sn. PocketQuake offers only stations
within 100 km, where Pn and Sn do not come first here.

Per model, the stage writes three files in `1.HypoInv/<model>/`. The **summary file** `.sum` has one line per
earthquake, the **archive file** `.arc` goes station by station, and the print file `.prt` is for people.

`st.pq.read_sum(PQ_SLUG, "kim1983")` reads a summary file. Its `num` counts the readings with final weight above
0.1. `erh` and `erz` are the **location errors** (horizontal and vertical, km). `qual` is a **location quality**
letter from A (best) to D; Klein (2002) calls it oversimplified, so read the numbers behind it.

## Part 2 · The first run and its log (about 60 min, task 1)

Your last picking run of M21 used a raised threshold (`PNPLUS_MIN_PROB`). This module needs PocketQuake's default
picks. So the first run repeats the picking stage, nothing changed, and then locates: stages `picking` to
`hypoinverse`.

Every run through `st.pq` adds one line to the run log `work/.sgtl/runs.jsonl`, in **JSON Lines** (one JSON object
per line). A line says what ran (`command`, `cluster`, `stage_from`, `through`) and whether it worked (`returncode`
0). It also holds when (`ts`, UTC) and the **SHA-256 fingerprint** of each product file (`products`). One changed byte
changes the fingerprint.

### M32-T1 · The run and its log · core · about 60 min

Write `run_products(log_path, slug, stage, stages)`. It returns one row per product of every successful `pq run` of
the cluster whose stages include `stage`; the docstring gives the rules. After the run, fill in `first_run`; its code
cell says how.

Example: a log with one matching run that logged one file gives one row, with `run` 0.

In [ ]:
# Before you run: will the picks of this run be the same, byte for byte, as those of a default picking run
# of M21?
st.predict("M32-T1", """

""")

In [ ]:
def run_products(log_path, slug, stage, stages):
    """One row per product of every successful 'pq run' of the cluster whose stages include the stage.

    log_path : str or pathlib.Path of a run log in JSON Lines
    slug     : the cluster name;  stage : one stage name;  stages : all stage names in run order (as st.pq.STAGES)
    A line matches when its command is "pq run", its cluster is slug (the whole name), its returncode is 0, and
    its stage_from comes at or before stage in stages and its through at or after it. Lines of other commands
    (such as "pq bootstrap") have no stage_from or through. Skip a line that is empty or not valid JSON.
    returns  : pandas DataFrame with the columns run, ts, product, sha256, in log order (a run's products in the
               order its line lists them). run numbers the matching runs 0, 1, 2, ...; a matching run without
               products keeps its number but adds no rows. No matching run: an empty DataFrame with these columns.
    """
    # Step 1: read the log line by line; skip a line that is empty or not valid JSON.
    # Step 2: keep the successful 'pq run' lines of this cluster whose range of stages holds the stage.
    # Step 3: number those runs 0, 1, 2, ... and add one row per product they logged.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T1: write your code here")

The run. Read the dry run first: it shows what would run and starts nothing. The picking takes one to two minutes
on one core.

**Watch out:** on the shared server use at most 4 cores (`PQ_CORES`), and one heavy run at a time.

In [ ]:
st.pq.run(PQ_SLUG, "picking", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB)          # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "picking", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
LOG = st.data.repo_path("work", ".sgtl", "runs.jsonl")

In [ ]:
# Fill in first_run = {"reader": run_products, "slug": PQ_SLUG, "fingerprints": {path: sha256}}, with one entry
# for every .sum, .arc and .prt file now in the model folders of HYP_DIR. Key each by its path as the log writes
# paths, "1.HypoInv/<model folder>/<file name>", and compute its SHA-256 now from the file's bytes:
# hashlib.sha256(path.read_bytes()).hexdigest().
# YOUR CODE HERE
raise NotImplementedError("M32-T1: write your code here")
print(f"reader {first_run['reader'].__name__}, slug {first_run['slug']!r}, "
      f"{len(first_run['fingerprints'])} fingerprints")

In [ ]:
st.check("M32-T1", answer=first_run)

What your last run logged, whether its picks fingerprint appears in an earlier run, and how many of your fingerprints
equal the logged ones:

In [ ]:
everything = run_products(LOG, PQ_SLUG, "picking", st.pq.STAGES)
last = everything[everything["run"] == everything["run"].max()]
print(last[["ts", "product", "sha256"]].assign(sha256=last["sha256"].str[:16] + "...").to_string(index=False))
picks_now = last.loc[last["product"].str.startswith("picks/"), "sha256"]
if len(picks_now):
    same = everything[(everything["sha256"] == picks_now.iloc[0]) & (everything["run"] < everything["run"].max())]
    print(f"\nthe picks fingerprint of this run appears in {same['run'].nunique()} earlier picking run(s) of "
          f"{PQ_SLUG!r}, logged at: {', '.join(same['ts'])}")
located = run_products(LOG, PQ_SLUG, "hypoinverse", st.pq.STAGES)
newest = located[located["run"] == located["run"].max()].set_index("product")["sha256"]
listed = [path for path in first_run["fingerprints"] if path in newest.index]
agree = sum(newest[path] == first_run["fingerprints"][path] for path in listed)
print(f"\nyour fingerprints: {len(first_run['fingerprints'])}; of files the last location run logged: "
      f"{len(listed)}, equal to the logged value: {agree}; of files it did not log: "
      f"{len(first_run['fingerprints']) - len(listed)}")

**Explain:** How does the log show, without opening a pick file, whether the picks of this run equal those of an
earlier picking run?

## Part 3 · The archive file, column by column (about 90 min, task 2)

The archive file shows how HYPOINVERSE got each location, station by station. Every value sits in fixed columns, and
neighbors often touch. Numbers carry no decimal point: F4.2 means 4 columns whose last two digits come after the
point, so `  -7` is −0.07 and `1234` is 12.34. F4.1 has one decimal, F3.0 none.

Two made-up station lines under a ruler of column numbers:

```
         1         2         3         4         5         6         7         8         9
1234567890123456789012345678901234567890123456789012345678901234567890123456789012345678901234
QQA1 XX  HHZ IPD1203103171246  605  -7124    0       0      0 0  0   0   0 23410600  0     211
QQA1 XX  HHZ    0203103171246    0   0  0 1895ES 2  31      0 0 62   0   0 23416100  0     211
```

The first holds a P reading (residual −0.07 s, weight 1.24), the second an S reading (0.31 s, 0.62). Both are 23.4 km
from the earthquake, at an azimuth of 211°.

### M32-T2 · The archive file, column by column · core · about 90 min

Write `read_arc(path)`. It turns a HYPOINVERSE archive file into a table with one row per P or S reading; the
docstring lists the columns. Header lines, closing lines and empty lines give no rows.

Example: the two lines above give a P row and then an S row of station `XX.QQA1`.

Not allowed: PocketQuake's code (`pipeline`, `pocketquake`), SKHASH or the course package.

In [ ]:
# Before you code: a colleague reads the station lines with line.split(). On which of the two lines above does
# that go wrong, and how?
st.predict("M32-T2", """

""")

In [ ]:
def read_arc(path):
    """The P and S readings of a HYPOINVERSE Y2000 archive file.

    path    : str or pathlib.Path of the .arc file
    A header line has the year in columns 1-4 and the earthquake's ID number in 137-146; a closing line is blank
    in 1-5; any other non-empty line is a station line. A station line gives one row for each remark (P, then S)
    that is not blank. Columns, counted from 1 (line[a - 1:b] in Python; a blank field is 0):
        1-5          station code                      51-54  F4.2  S residual (s)
        6-7          network code                      64-66  F3.2  S weight used
        14-15        P remark, such as IP              75-78  F4.1  epicentral distance (km)
        35-38  F4.2  P residual (s)                    92-94  F3.0  azimuth (degrees)
        39-41  F3.2  P weight used
        47-48        S remark, such as ES
    returns : pandas DataFrame with event (int, the ID number of the header line above), station
              ('network.station', such as 'XX.QQA1'), phase ('P' or 'S'), residual_s, weight, distance_km and
              azimuth_deg (floats), in file order
    """
    # Step 1: decide the kind of each line: header (year in columns 1-4), closing (columns 1-5 blank), station.
    # Step 2: on a header line, remember the ID number of columns 137-146.
    # Step 3: on a station line, read each field from its columns with its implied decimal point (blank = 0), and
    #         add a P row when columns 14-15 are not blank and an S row when columns 47-48 are not blank.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T2: write your code here")

In [ ]:
st.check("M32-T2", func=read_arc, slug=PQ_SLUG)

Your two archive files in numbers, and the readings that HYPOINVERSE gave no weight:

In [ ]:
MODELS = ("kim1983", "kim2011")
ARC = {m: sorted((HYP_DIR / m).glob("*.arc"))[0] for m in MODELS}
phases = {m: read_arc(ARC[m]) for m in MODELS}
for m in MODELS:
    table = phases[m]
    zero = table[table["weight"] == 0]
    print(f"{m}: {len(table)} readings ({(table['phase'] == 'P').sum()} P, {(table['phase'] == 'S').sum()} S) "
          f"at {table['station'].nunique()} stations; weight above 0.1: {(table['weight'] > 0.1).sum()}; "
          f"weight 0: {len(zero)}")
zero83 = phases["kim1983"][phases["kim1983"]["weight"] == 0]
print("\nkim1983, the readings with weight 0:")
print(zero83[["event", "station", "phase", "residual_s", "distance_km"]].to_string(index=False))

**Explain:** A **phase misidentification** is a pick with the wrong phase label. Which readings of weight 0 (printed
above) look like one? (S-P grows by about 1 s per 8 km.)

## Part 4 · Three locations of each earthquake (about 75 min, task 3)

You now have three kinds of locations: PocketQuake's HYPOINVERSE, the KMA catalog, and your two locations of M31. Both
HYPOINVERSE runs share the program, the model (kim1983) and the commands: M31 sent PocketQuake's template line by
line. Only the picks, their stations and their weights differ.

To compare two lists, pair their earthquakes by origin time. **Watch out:** your STP folder's `event_catalog.csv` is
in KST (UTC + 9 h), but the course's KMA catalog and HYPOINVERSE's files are in UTC. Compare only aware times, which
carry their time zone. The next cell puts each list in one form: `time`, `lat`, `lon`, `depth`.

In [ ]:
if not Path(M31_FILE).is_file():
    raise FileNotFoundError(f"{M31_FILE} is missing. Part 4 compares HYPOINVERSE with your own locations of the "
                            "mainshock: finish task M31-T6 of module M31 first (its cells save that table), or set "
                            "M31_FILE in the parameters cell to where you saved it.")


def sum_table(summary):
    """st.pq.read_sum's table with aware UTC times (the form match_locations takes)."""
    out = summary.copy()
    out["time"] = pd.to_datetime([str(t) for t in summary["time"]], utc=True)
    return out


def catalog_table(catalog, time="origin_utc", lat="latitude", lon="longitude", depth="depth_km"):
    """A catalog (such as KMA's) with the columns time (aware), lat, lon, depth."""
    out = catalog.rename(columns={time: "time", lat: "lat", lon: "lon", depth: "depth"}).copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)
    return out


hyp83 = sum_table(st.pq.read_sum(PQ_SLUG, "kim1983"))
kma_tab = catalog_table(kma)
m31_raw = pd.read_csv(M31_FILE)
mine = m31_raw[m31_raw["location"].isin(["your geiger", "run 1"])].reset_index(drop=True)
if mine.empty:
    raise ValueError(f"{M31_FILE} has no rows 'your geiger' or 'run 1'; rerun the cells of M31's last part")
m31_origin = kma.loc[kma["event_id"] == M31_EVENT_ID, "origin_utc"].iloc[0]
m31_tab = pd.DataFrame({"location": mine["location"].replace({"run 1": "HYPOINVERSE on your picks"}),
                        "time": m31_origin + pd.to_timedelta(mine["origin - KMA (s)"], unit="s"),
                        "lat": mine["latitude"], "lon": mine["longitude"], "depth": mine["depth (km)"]})
print(f"tables ready: HYPOINVERSE kim1983 {len(hyp83)} rows, KMA {len(kma_tab)} rows, your M31 table "
      f"{len(m31_tab)} row(s)")

### M32-T3 · Three locations of each earthquake · core · about 75 min

Write `match_locations(ref, other, max_dt_s)`. It pairs each earthquake of `ref` with the one of `other` closest in
origin time. Then it measures how far apart the two locations are; the docstring gives the details.

Example: ref at `2031-03-17 03:46:10+00:00` (35.10 N, 127.20 E, 8.0 km) and other at
`2031-03-17 12:46:11.5+09:00` (35.11 N, 127.20 E, 9.5 km) are a pair with `dt_s` 1.5, `dist_km` about 1.11 and
`ddepth_km` 1.5.

Not allowed: the course package.

In [ ]:
# Before you code: how far apart do you expect HYPOINVERSE's (kim1983) and KMA's epicenters to be?
st.predict("M32-T3", """

""")

In [ ]:
def match_locations(ref, other, max_dt_s):
    """Pair each earthquake of ref with the one of other closest in origin time (within max_dt_s).

    ref, other : DataFrames with the columns time (aware pandas times, any time zone), lat, lon (degrees) and depth
                 (km); any index. A table whose times carry no time zone cannot be compared with anything: raise a
                 ValueError that says so.
    max_dt_s   : the largest origin-time difference of a pair (s)
    returns    : DataFrame with the index of ref (same rows, same order) and the columns
        match      the index label of the partner in other: the row with the closest origin time, if it is at most
                   max_dt_s away; otherwise None
        dt_s       the partner's time minus ref's time (s)
        dist_km    the epicentral distance between the two epicenters along the Earth's surface (km; a sphere of
                   radius 6371 km is accurate enough)
        ddepth_km  the partner's depth minus ref's depth (km)
        (NaN in the three numbers when there is no partner)
    """
    # Step 1: refuse times without a time zone; then put both tables on one clock (UTC).
    # Step 2: for every ref row, find the other row closest in time; keep it only within max_dt_s.
    # Step 3: for a pair, compute the time difference, the distance along the surface and the depth difference.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T3: write your code here")

In [ ]:
st.check("M32-T3", func=match_locations, slug=PQ_SLUG)

PocketQuake's HYPOINVERSE (kim1983) against KMA and against your two M31 locations. `dmin_km`, the **distance to the
nearest station** with a weighted reading, comes from your archive table of task 2.

In [ ]:
dmin83 = phases["kim1983"][phases["kim1983"]["weight"] > 0.1].groupby("event")["distance_km"].min()
view = hyp83.assign(dmin_km=hyp83["id"].map(dmin83))
print("HYPOINVERSE, kim1983:")
print(view[["id", "time", "lat", "lon", "depth", "num", "gap", "dmin_km", "rms", "erh", "erz", "qual"]]
      .to_string(index=False))
vs_kma = match_locations(hyp83, kma_tab, MAX_DT_S)
vs_kma.insert(0, "kma_event", [kma_tab.loc[m, "event_id"] if m is not None else None for m in vs_kma["match"]])
print("\nKMA minus HYPOINVERSE (kim1983):")
print(vs_kma.drop(columns="match").round(3).to_string())
vs_m31 = match_locations(m31_tab, hyp83, MAX_DT_S)
vs_m31.insert(0, "location", m31_tab["location"])
print("\nPocketQuake's HYPOINVERSE (kim1983) minus your M31 locations of the mainshock:")
print(vs_m31.drop(columns="match").round(3).to_string())
lat_c = float(hyp83["lat"].mean())
print(f"\nKMA's rounding at {lat_c:.1f} N: 0.01 degree is {0.01 * 111.19:.2f} km north-south and "
      f"{0.01 * 111.19 * np.cos(np.radians(lat_c)):.2f} km east-west")
vs_kma.to_csv(Path(RESULT_DIR) / "M32_hypoinverse_vs_kma.csv")

A map: KMA's epicenters (gray squares; several fall on the same rounded point), PocketQuake's HYPOINVERSE (red
circles) and your two M31 locations (blue).

In [ ]:
import pygmt

region = [128.07, 128.13, 36.485, 36.525]
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=region, projection="M10c", frame=["af", "+tSangju earthquakes located three ways"])
    fig_map.plot(x=kma_tab.loc[vs_kma["match"].dropna(), "lon"], y=kma_tab.loc[vs_kma["match"].dropna(), "lat"],
                 style="s0.35c", fill="gray70", pen="0.5p,gray30", label="KMA catalog+S0.25c")
    fig_map.plot(x=hyp83["lon"], y=hyp83["lat"], style="c0.2c", fill="red3", pen="0.4p,black",
                 label="HYPOINVERSE kim1983+S0.2c")
    for name, style, text in (("your geiger", "a0.4c", "your Geiger location (M31)+S0.3c"),
                              ("HYPOINVERSE on your picks", "d0.3c", "HYPOINVERSE on your picks (M31)+S0.25c")):
        rows = m31_tab[m31_tab["location"] == name]
        if len(rows):
            fig_map.plot(x=rows["lon"], y=rows["lat"], style=style, fill="dodgerblue", pen="0.5p,black", label=text)
    fig_map.legend(position="jTL+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(region, km=1))
fig_map.savefig(Path(FIGURE_DIR) / "three_locations.png", dpi=150)
fig_map.show()

**Explain:** Which earthquake's depth differs most between HYPOINVERSE and KMA, and do its `erz`, `num`, `gap` and
`dmin_km` (printed above) show why?

## Part 5 · An unchanged rerun and a second velocity model (about 75 min, task 4)

Two habits for a production tool, as in M31: an unchanged rerun (the same files, byte for byte?), then one change.
The change here is the velocity model: the same picks and settings with kim1983 and with kim2011. The stage writes
both, so you read the other folder.

The cells copy the folder `1.HypoInv` of your first run to `results/hypinv/first`. Then they rerun the location stage
alone (about a second) and copy the folder to `results/hypinv/rerun`. A run overwrites the folder, so only a copy
keeps the earlier result.

### M32-T4 · An unchanged rerun and a second velocity model · core · about 75 min

Write `differing_lines(path_a, path_b)`: the lines at which two text files differ, with their line numbers. After
the rerun, fill in `rerun_report`; its code cell says how.

Example: files with the lines `a`, `b`, `c` and `a`, `x`, `c`, `d` give `[(2, "b", "x"), (4, None, "d")]`.

Not allowed: `difflib`, `filecmp` or the course package.

In [ ]:
# Before you run: from kim1983 to kim2011, with the same picks, will the epicenters or the depths move more?
st.predict("M32-T4", """

""")

In [ ]:
def differing_lines(path_a, path_b):
    """The lines at which two text files differ: a list of (line_number from 1, line_a, line_b).

    path_a, path_b : str or pathlib.Path of the two files
    returns        : list of tuples (line_number, line_a, line_b), one for every line number at which the files
                     differ, in order; line numbers count from 1 and lines come without their end-of-line
                     characters; a line that exists in only one file has None for the other; [] for two files that
                     hold the same lines
    """
    # Step 1: read both files as lines without their end-of-line characters.
    # Step 2: walk through the line numbers of the longer file; a missing line is None.
    # Step 3: keep the numbers at which the two lines differ, counting from 1.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T4: write your code here")

In [ ]:
COPIES = Path(RESULT_DIR) / "hypinv"


def keep_copy(label):
    """Copy the cluster's 1.HypoInv folder to results/hypinv/<label> (the next run overwrites the original)."""
    target = COPIES / label
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(HYP_DIR, target)
    print(f"copied {sum(1 for p in target.rglob('*') if p.is_file())} files to {target}")
    return target


first_dir = keep_copy("first")
st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_dir = keep_copy("rerun")

In [ ]:
# Fill in rerun_report, a dict with the keys
#   differ        your differing_lines function;  slug  PQ_SLUG
#   first, rerun  the folders of the two copies
#   fingerprints  {"first": {...}, "rerun": {...}}: for each copy, the path of every file inside it (relative to the
#                 copy, written with /, such as "kim1983/<file name>") -> the SHA-256 of the file's bytes (64 hex
#                 letters, as hexdigest() writes them)
#   model_change  your match_locations with ref = the kim1983 summary and other = the kim2011 summary of the first
#                 copy (st.pq.read_sum takes a path to a .sum file; sum_table makes its times aware)
# YOUR CODE HERE
raise NotImplementedError("M32-T4: write your code here")
print({k: v for k, v in rerun_report.items() if k in ("slug", "first", "rerun")})

In [ ]:
st.check("M32-T4", answer=rerun_report)

Which files differ between the two copies, what differs in them, and how the locations move from kim1983 to kim2011
(kim2011 minus kim1983):

In [ ]:
prints = rerun_report["fingerprints"]
differ_files = [rel for rel, digest in prints["first"].items() if prints["rerun"].get(rel) != digest]
print(f"files with the same SHA-256 in both copies: {len(prints['first']) - len(differ_files)} of "
      f"{len(prints['first'])}; files that differ: {differ_files}")
for rel in differ_files:
    if (rerun_dir / rel).is_file():
        for number, a, b in differing_lines(first_dir / rel, rerun_dir / rel):
            print(f"{rel}, line {number}:\n  first: {a}\n  rerun: {b}")
both = {m: sum_table(st.pq.read_sum(sorted((first_dir / m).glob("*.sum"))[0])) for m in MODELS}
change = rerun_report["model_change"].copy()
change.insert(0, "id", both["kim1983"]["id"].to_numpy())
for col in ("depth", "rms", "erz", "qual"):
    change[f"{col}_83"] = both["kim1983"][col].to_numpy()
    change[f"{col}_11"] = both["kim2011"][col].to_numpy()
print("\nkim2011 minus kim1983:")
print(change.drop(columns="match").round(3).to_string(index=False))
for name in MODELS:
    layers = st.data.velocity_model(name)
    print(f"\n{name}:", "; ".join(f"from {r.top_km:g} km: Vp {r.vp:.2f}, Vs {r.vs:.2f} km/s"
                                  for r in layers.itertuples()))

fig, ax = plt.subplots(figsize=(7.0, 3.6))
x = np.arange(len(change))
ax.plot(x, change["depth_83"], "o", color="tab:red", label="kim1983")
ax.plot(x, change["depth_11"], "s", color="tab:blue", label="kim2011")
ax.set_xticks(x, [str(i) for i in change["id"]], rotation=30)
ax.invert_yaxis()
ax.set_xlabel("Earthquake (PocketQuake id)")
ax.set_ylabel("Depth (km)")
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "depth_two_models.png", dpi=150)
plt.show()

**Explain:** From kim1983 to kim2011, what moved more, the epicenters or the depths (printed above), and why?

## Part 6 · Early and late stations (about 75 min, task 5)

A **station delay** is a constant time that a locator adds at one station. It stands for what a one-dimensional
model misses near that station: slow sediments or fast rock below it, or its height. A station whose residuals are
consistently late or early, over many earthquakes, is a candidate for a delay.

"Consistently" needs a measure. For $n$ residuals with sample standard deviation $s$, the standard error of their
mean is $s/\sqrt{n}$. A mean more than two standard errors from zero is unlikely to be chance.

### M32-T5 · Early and late stations · core · about 75 min

Write `station_terms(phases, min_events)`. For every station and phase, it returns the number of used readings, their
mean residual, its standard error and a label: `"late"`, `"early"` or `""`. The docstring gives the rules.

Example: P residuals of 0.20, 0.30 and 0.25 s (weights 0.9, 1.2, 0.8) and one of 8.0 s (weight 0), with
`min_events` 3. They give n 3, mean_s 0.25, se_s 0.0289 and the label "late".

Not allowed: the course package.

In [ ]:
# Before you code: under kim1983, will a few stations stand out, or will most stations share one pattern?
st.predict("M32-T5", """

""")

In [ ]:
def station_terms(phases, min_events):
    """Mean residual, standard error and label of every station and phase with at least min_events used readings.

    phases     : DataFrame like those of read_arc (at least event, station, phase, residual_s and weight); any index
                 and row order
    min_events : the smallest number of used readings for a row
    Use only the readings that HYPOINVERSE used: weight above 0.1 (strictly above; the rule of the summary's num).
    returns    : DataFrame with one row per station and phase that has at least min_events used readings, sorted by
                 station, then phase, with the columns
        station, phase
        n        the number of used readings
        mean_s   their mean residual (s)
        se_s     its standard error: the sample standard deviation (with n - 1) divided by the root of n
        label    'late' when mean_s - 2 se_s > 0, 'early' when mean_s + 2 se_s < 0, else ''
    """
    # Step 1: keep the readings with weight above 0.1.
    # Step 2: per station and phase: the number, the mean and the sample standard deviation (n - 1) of the residuals.
    # Step 3: keep the groups with at least min_events readings; the standard error is the deviation over the root
    #         of n; label by the two-standard-error rule; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M32-T5: write your code here")

In [ ]:
st.check("M32-T5", func=station_terms, reader=read_arc, slug=PQ_SLUG)

Your station terms in both models, the largest under kim2011, and the kim2011 terms against station height. The
layer model puts every station at height 0.

In [ ]:
from scipy.stats import linregress

terms = {m: station_terms(phases[m], MIN_EVENTS) for m in MODELS}
for m in MODELS:
    t = terms[m]
    print(f"{m}: {len(t)} station-phases with at least {MIN_EVENTS} readings")
    print(pd.crosstab(t["phase"], t["label"].replace("", "neither")).to_string(), "\n")
big = terms["kim2011"].reindex(terms["kim2011"]["mean_s"].abs().sort_values(ascending=False).index).head(10)
print("kim2011, the ten largest station terms:")
print(big.round(3).to_string(index=False))
terms["kim2011"].to_csv(Path(RESULT_DIR) / "M32_station_terms_kim2011.csv", index=False)

stations = pd.read_csv(RUN_DIR / "station_table" / "used_stations_100km.csv")
height_km = pd.Series(stations["Elevation"].to_numpy() / 1000.0,
                      index=stations["Network"] + "." + stations["Code"])
top = st.data.velocity_model("kim2011").iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, phase, speed in zip(axes, ("P", "S"), (top.vp, top.vs)):
    t = terms["kim2011"][terms["kim2011"]["phase"] == phase]
    h = t["station"].map(height_km)
    fit = linregress(h, t["mean_s"])
    print(f"kim2011 {phase}: slope {fit.slope:.3f} +- {fit.stderr:.3f} s per km of height (r = {fit.rvalue:.2f}, "
          f"{len(t)} stations); a vertical ray in the top layer: {1 / speed:.3f} s per km")
    ax.errorbar(h, t["mean_s"], yerr=t["se_s"], fmt="o", ms=4, color="tab:blue", ecolor="0.6", label="Station term")
    hh = np.linspace(0, float(h.max()), 2)
    ax.plot(hh, fit.intercept + fit.slope * hh, color="tab:red", label="Least-squares line")
    ax.plot(hh, fit.intercept + hh / speed, "--", color="0.3", label="Vertical ray, top layer")
    ax.axhline(0, color="0.8", lw=0.8)
    ax.set_title(f"{phase} waves, kim2011", fontsize=10)
    ax.set_xlabel("Station height (km)")
axes[0].set_ylabel("Mean residual (s)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "station_terms_height.png", dpi=150)
plt.show()

**Explain:** What pattern do the kim1983 labels show for P and for S (printed above), and what does it say about the
model?

## What you made, and what comes next

Your location folders are in `results/hypinv/`, your tables in `results/` and your figures in `figures/`. M41
relocates these earthquakes relative to each other with HypoDD, starting from your kim1983 archive file.

Station coordinates come from STP's SAC headers. HYPOINVERSE uses arrival times only, so the course's station
metadata (StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong National University) is not needed here. The KMA catalog
is used under the Korea Open Government License Type 1 (attribution).

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M32")

**Optional:** repeat task 5 with `MIN_EVENTS = 3` and `5`. Which labels are stable?

**Read more:** Klein (2002), the HYPOINVERSE manual; Husen and Hardebeck (2010), location errors; Kissling et al.
(1994), station corrections.